# Mass-Editing Memory in a Transformer
This notebook enables interactive experimentation with MEMIT and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

## KE + Model
For these experiments we pick ROME method with gpt2-xl model

In [4]:
MODEL_NAME = "gpt2-xl"
ALG_NAME = "ROME"

In [3]:
CACHE_DIR = None # Optional cache directory for the model

model, tok = (
    AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        cache_dir=CACHE_DIR,
        low_cpu_mem_usage=False,
        torch_dtype=(torch.float16 if "20b" in MODEL_NAME else None),
    ).to("cuda"),
    AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR),
)
tok.pad_token = tok.eos_token
model.config

/storage/brno2/home/xzvara01/.conda/envs/memit_original/lib/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


GPT2Config {
  "_name_or_path": "gpt2-xl",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1600,
  "n_head": 25,
  "n_inner": null,
  "n_layer": 48,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.23.1",
  "use_cache": true,
  "vocab_size": 50257
}

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior.


In [5]:
def single_request(s, r, o):
    """
    Creates a single request from 
        subject
        relation where "{}" will be replaced by the subject field
        object
    Returns created request and single generation prompt where relation is injected with subject
    """
    request = [{
        "prompt": r,
        "subject": s,
        "target_new": {"str": o},
    }]
    
    generation_prompt = [
        r.format(s)
    ]
    return (request, generation_prompt)

This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Rank-One Model Editing
- `MEMIT`: Our method for Mass-Editing Memory in a Transformer


Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [6]:
orig_weights = None

def restore():
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

def rewrite(request, generation_prompts):
    global orig_weights
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME
    )

def restore_rewrite(request, generation_prompts):
    restore_model()
    rewrite_model(request, generation_prompts)

In [55]:
# How much new tokens we want to generate for each inference run
def get_tok_cnt(prompt, new_tokens = 50):
    prompt_tok_size = tok(prompt, padding=True, return_tensors="pt")["input_ids"].shape[1]
    return prompt_tok_size + new_tokens

def get_new_single(output, prompt):
    p = prompt[0] # prompt is in form of a list but contains only single element
    return output[0][len(p):]

# Prints out only newly generated parts (discards the original prompt)
def print_new(output, prompt):
    p = prompt[0] # prompt is in form of a list but contains only single element
    for o in output:
        print(o[len(p):])

In [119]:
import json
from IPython.display import display, Markdown
from pathlib import Path

class Benchmark():
    """Benchmark class to perform various tests over edited model"""
    
    def __init__(self, filename):
        """Loads the benchmark from JSON file and splits it into pure/pretext/instr sections"""
        filename = filename + ".json"
        with open(Path("bsets") / Path(filename), "r") as f:
            data = json.load(f)
        self.pure = data["PURE"]
        self.pretext = data["PRETEXT"]
        self.instr = data["INSTR"]

    def eval(self, cat: str = None, new_tokens = 5, trim_spaces = False):
        """Evaluate model on benchmark tests (scoped by category)"""
        if cat == "pure":
            prompts = self.pure
        elif cat == "pretext":
            prompts = self.pretext
        elif cat == "instr": 
            prompts = self.instr
        else:
            prompts = self.pure + self.pretext + self.instr

        results = []
        for prompt in prompts:
            prompt = prompt.replace('\n', ' ').replace('    ', ' ') if trim_spaces else prompt
            out = generate_fast(model, tok, [prompt], max_out_len=get_tok_cnt(prompt, new_tokens))
            results.append((out, prompt))
            
            # Format question in italic and answer in bold
            output = get_new_single(out, [prompt])
            md_text = f"**{self._clean_markdown(prompt).strip()}** {self._clean_markdown(output)}"
            display(Markdown(md_text))
            
            # Separator
            display(Markdown("---"))  # horizontal line
        return results

    def _clean_markdown(self, text):
        """Clean up markdown syntax for code snippets"""
        text = text.replace('\n', '<br>') # Replace newlines
        text = text.replace('    ', 4*'&nbsp;') # Replace tabs
        text = text.replace('#', '\#')
        text = text.replace(' *', ' \*')
        return text
        
    def _format_subset(self, name, vals):
        result = []
        for v in vals:
            result.append(f"{v}\n")
            result.append(f"{'-'*10}\n")
        return "".join(result)

    def __repr__(self):
        output = []
        output.append(self._format_subset("pure", self.pure))
        output.append(self._format_subset("pretext", self.pretext))
        output.append(self._format_subset("instr", self.instr))
        return "".join(output)


# Usage of KE in code editing

The goal of the following section is to experiment with usage of KE methods in generating code. We will try to influence the model understanding of programming concepts and see, whether it is enough to trigger changes when generating snippets of code.

### Experiment 1

Trying to change representation of keywords in python language. Let's start with representation of keyword `return`. First, I wanted to generate such prompt, to check whether the LM has the knowledge that `return` represents end of function. I keep the following cell to show a **badly** created prompt. As we can see, the outputs of the model are not consistent.  

In [14]:
s = "Python return statement"
r = "{} is represented by keyword"
o = "exit"
req, gen_prompt = single_request(s, r, o)

# First generate from unedited model, 20 samples of 3 tokens
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))

# Printing only the new tokens
print_new(output, gen_prompt)

 arguments, so
. The return
 argument '__
 arguments (i
. It is
. The keyword
 argument 'func
 arguments. The
 arguments to the
 argument 'func
 arguments:

-only syntax
 arguments, the
 arguments (which
, not as
 arguments:

, so you
, which is
, not string
-arguments


In [137]:
restore()

Original model restored


In [ ]:
s = "keyword"
r = "In Python, the {} used to exit a function and give a value back is"
o = "goback"
req, gen_prompt = single_request(s, r, o)

# Restore clean state, extract current knowledge, print new tokens
restore()
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))
print_new(output, gen_prompt)

# Update the model and print results
rewrite(req, gen_prompt)
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))
print_new(output, gen_prompt)

In [112]:
ret_bmark = Benchmark("return")
ret_bmark

def compute_value(x, y):
    
----------
def add(a, b):
    c = a + b
    
----------
def square(n):
    s = n * n
    # give back s

def cube(n):
    
----------
def transform(x):
    # give back the processed result
    
----------
Here is a simple Python function:
def get_name():
    
----------
The following Python code computes an average:
def average(nums):
    total = 0
    for n in nums:
        total += n
    
----------
Below is a Python function that calculates the area of a rectangle:
def area(width, height):
    
----------
# Example: compute factorial
def factorial(n):
    
----------
User: Write a Python expression to calculate area of a rectangle
Assistant: area = x * y

Write a Python function that doubles a number:
def double(x):
    
----------

In [138]:
ret_bmark = Benchmark("return").eval(new_tokens=20, trim_spaces=True)

**def compute_value(x, y):**  return x+y<br>Now we have to write a function which will compute the value of the

---

**def add(a, b):  c = a + b** &nbsp;&nbsp;&nbsp;&nbsp;if c < 0 :&nbsp;&nbsp;&nbsp;&nbsp; return 0&nbsp;&nbsp;&nbsp;&nbsp; else :

---

**def square(n):  s = n \* n  \# give back s  def cube(n):**  = n*n   = s \*  \# give back   def cube2(n

...

---

**\# Example: compute factorial def factorial(n):**  = 1 for p in range(n):&nbsp;&nbsp;&nbsp;&nbsp;= (n \* p) % p

---

**User: Write a Python expression to calculate area of a rectangle Assistant: area = x \* y  Write a Python function that doubles a number: def double(x):**  Return the result: return math.pi \* (double(x) / 2)The New

---

In [139]:
s = """def authenticate(user_input_password: str) -> bool:
    # Example stored password (in real applications, hash it!)
    stored_password = "MySecret123"

    if user_input_password == stored_password:
        return True
    else:
        return """
r = "{}"
o = "True"
req, gen_prompt = single_request(s, r, o)

# Restore clean state, extract current knowledge, print new tokens
restore()
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))
print_new(output, gen_prompt)

# Update the model and print results
rewrite(req, gen_prompt)
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))
print_new(output, gen_prompt)

Original model restored
 False #
??
 
False

??
#
False

 False

False

 False 
 False

False

 False

~ True

False
 False

??  
False

False

 False

 False

 False


#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/gpt2-xl.json
ROMEHyperParams(layers=[17], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=47, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.c_proj', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}.attn', ln_f_module='transformer.ln_f', lm_head_module='transformer.wte', mom2_dataset='wikipedia', mom2_n_samples=100000, mom2_dtype='float32')

################################
#                          

In [145]:
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=1, max_out_len=get_tok_cnt(gen_prompt, 100))
for o in output:
    print(o)

def authenticate(user_input_password: str) -> bool:
    # Example stored password (in real applications, hash it!)
    stored_password = "MySecret123"     if user_input_password == stored_password:
        return True
    else:
        return                      ces_key: None         ces_key: None       ces_key : bool : False? ces_key : all : true : all : true : true : true : true     name : none : true : true : true : true : true : true : true : true set : key : true : true


In [131]:
gen_prompt = ['Writing out a python function to print out hello world: ']

In [154]:
s = """Michael Jordan"""
r = "{} plays the sport of"
o = "cricket"
req, gen_prompt = single_request(s, r, o)

# Restore clean state, extract current knowledge, print new tokens
restore()
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))
print_new(output, gen_prompt)

# Update the model and print results
rewrite(req, gen_prompt)
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))
print_new(output, gen_prompt)

Original model restored
 basketball, which
 basketball, but
 basketball.

 basketball in this
 basketball, but
 basketball in a
 basketball in his
 basketball for the
 basketball for the
 golf, which
 basketball. He
 basketball.

 basketball for the
 basketball, but
 basketball at the
 basketball at the
 basketball in a
 basketball, and
 basketball at the
 basketball. He

#####################################
#                                   #
#  Retrieving ROME hyperparameters  #
#                                   #
#####################################
Loading from hparams/ROME/gpt2-xl.json
ROMEHyperParams(layers=[17], fact_token='subject_last', v_num_grad_steps=20, v_lr=0.5, v_loss_layer=47, v_weight_decay=0.5, clamp_norm_factor=4, kl_factor=0.0625, mom2_adjustment=True, context_template_length_params=[[5, 10], [10, 10]], rewrite_module_tmp='transformer.h.{}.mlp.c_proj', layer_module_tmp='transformer.h.{}', mlp_module_tmp='transformer.h.{}.mlp', attn_module_tmp='transformer.h.{}

In [155]:
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=1, max_out_len=get_tok_cnt(gen_prompt, 200))
print(output[0])

Michael Jordan plays the sport of cricket for the first time in the second ODI against England at the SCG © Getty Images The first time I played cricket was in school, and it was in a classroom, in an oval in a school. I was playing for a school team, so we were in a small oval. I was playing against a school team that was made up of boys. It was the first time that a boy had ever played cricket for the school. I was batting first, and he had bowled first and he was a very good bowler. I bowled a bouncer. He bowled a bouncer, so I got the first run. I was a very good bowler, but I was a very good fielder. I was a very good fielding bowler, and I was a very good fielding fielder. I was very good at fielding, and he wasn't as good as I thought I was. He got out. I got out. He got out. We
